# `ModelFallbackMiddleware`

Middleware that automatically retries failed model calls using alternative fallback models.

The primary model is configured separately in `create_agent`. If the primary model raises an exception, the middleware tries each configured fallback model in order until one succeeds. If every model fails, the final exception is re-raised.

When switching from an Anthropic model to a non-Anthropic fallback, the middleware removes Anthropic-specific `cache_control` markers from the request to prevent provider compatibility errors.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
ModelFallbackMiddleware(
    first_model: str | BaseChatModel, # First fallback model
    *additional_models: str | BaseChatModel # Additional fallbacks in order
)
```

### Parameters

* `first_model` — First fallback model to try after the primary model fails.
  * May be a model identifier string such as `"openai:gpt-5.5"`.
  * May also be an initialized `BaseChatModel` instance.
* `*additional_models` — Additional fallback models tried sequentially after `first_model`.
  * Each value may be either a model identifier string or a `BaseChatModel` instance.

String model identifiers are initialized internally using `init_chat_model`.

## Attributes

* `models` — Ordered list of initialized fallback model instances.
  * Type: `list[BaseChatModel]`
  * Models are stored in the same order in which they were passed to the constructor.

## Methods

1. `wrap_model_call`: Executes a synchronous model call and tries fallback models when errors occur.
   * The original request uses the primary model configured in `create_agent`.
   * Fallback models are tried one by one in the configured order.
   * Returns immediately after the first successful model response.
   * Re-raises the last exception when every model fails.
   * Removes Anthropic `cache_control` markers before using a non-Anthropic fallback.
   - **Syntax:**
     ```python
     wrap_model_call(
         self,
         request: ModelRequest[ContextT], # Initial model request
         handler: Callable[
             [ModelRequest[ContextT]],
             ModelResponse[ResponseT]
         ] # Function that executes the model request
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

2. `awrap_model_call`: Asynchronous version of `wrap_model_call`.
   * Awaits the primary model call first.
   * Tries fallback models sequentially when the primary model fails.
   * Returns after the first successful asynchronous model response.
   * Re-raises the last exception when every model fails.
   * Applies the same provider-specific request sanitization as the synchronous method.
   - **Syntax:**
     ```python
     async def awrap_model_call(
         self,
         request: ModelRequest[ContextT], # Initial model request
         handler: Callable[
             [ModelRequest[ContextT]],
             Awaitable[ModelResponse[ResponseT]]
         ] # Async function that executes the model request
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

## Fallback Execution Order

For the following configuration:

```python
fallback = ModelFallbackMiddleware(
    "openai:gpt-5.5",
    "anthropic:claude-sonnet-4-5-20250929"
)

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[fallback]
)
```

The models are attempted in this order:

```text
1. Primary model configured in create_agent
2. First fallback model
3. Second fallback model
4. Remaining fallback models, if any
```

A fallback is attempted only when the previous model call raises an exception.

## Anthropic Cache-Control Handling

Anthropic prompt-caching middleware may add provider-specific `cache_control` markers to model requests.

These markers are valid for Anthropic-compatible fallback models but may cause errors when sent to providers such as OpenAI or Gemini.

Before a non-Anthropic fallback attempt, the middleware removes `cache_control` from:

* Model settings
* System-message content blocks
* Request-message content blocks
* Tool dictionaries
* `BaseTool.extras`
* Nested `extras` and `metadata` dictionaries inside content blocks

The markers are preserved when the fallback model reports one of these Anthropic-compatible `_llm_type` values:

```python
{
    "anthropic-chat",
    "anthropic-bedrock-chat",
    "anthropic-chat-vertexai",
}
```

## Internal Helper Functions

### `_sanitize_content_blocks`

Removes Anthropic cache markers from dictionary-based message content blocks.

```python
_sanitize_content_blocks(
    content: str | list[str | dict[str, Any]]
) -> str | list[str | dict[str, Any]]
```

### `_sanitize_system_message`

Returns a system message without Anthropic cache markers.

```python
_sanitize_system_message(
    system_message: SystemMessage | None
) -> SystemMessage | None
```

### `_sanitize_messages`

Sanitizes all request messages.

```python
_sanitize_messages(
    messages: list[AnyMessage]
) -> list[AnyMessage]
```

### `_sanitize_tools`

Removes cache markers from both `BaseTool` instances and dictionary-style tool definitions.

```python
_sanitize_tools(
    tools: list[BaseTool | dict[str, Any]]
) -> list[BaseTool | dict[str, Any]]
```

### `_sanitize_request_for_fallback`

Creates an overridden request containing only the fields that required sanitization.

```python
_sanitize_request_for_fallback(
    request: ModelRequest[ContextT]
) -> ModelRequest[ContextT]
```

### `_sanitize_message`

Sanitizes one message and reports whether it changed.

```python
_sanitize_message(
    message: AnyMessage
) -> tuple[AnyMessage, bool]
```

### `_sanitize_base_tool`

Removes `cache_control` from a `BaseTool` object's `extras`.

```python
_sanitize_base_tool(
    tool: BaseTool
) -> tuple[BaseTool, bool]
```

### `_sanitize_dict_tool`

Removes `cache_control` from a dictionary-style tool payload and its nested `extras`.

```python
_sanitize_dict_tool(
    tool: dict[str, Any]
) -> tuple[dict[str, Any], bool]
```

### `_without_cache_control`

Returns a dictionary without its top-level `cache_control` key and indicates whether a change occurred.

```python
_without_cache_control(
    payload: dict[str, Any]
) -> tuple[dict[str, Any], bool]
```

### `_without_cache_control_from_content_block`

Removes `cache_control` from a content block and from nested `extras` and `metadata` dictionaries.

```python
_without_cache_control_from_content_block(
    block: dict[str, Any]
) -> tuple[dict[str, Any], bool]
```

### `_supports_anthropic_cache_control`

Checks whether a model accepts Anthropic-compatible cache markers by examining its `_llm_type`.

```python
_supports_anthropic_cache_control(
    model: BaseChatModel
) -> bool
```

## Example

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

fallback = ModelFallbackMiddleware(
    "openai:gpt-5.5",
    "anthropic:claude-sonnet-4-5-20250929"
)

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[fallback]
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain fallback models."}]}
)
```

If the primary model call fails, the middleware tries the configured fallback models in sequence.